# AI-Powered Intrusion Detection Solution - CLO4 Project

## 1) Introduction & Problem Definition

**Organization Context**: SecureNet Corp. needs to augment firewall-based controls with behavior-based ML detection.

**Security Objective**: Build a proof-of-concept NIDS that classifies traffic as **Benign** or **Malicious** while emphasizing low false negatives.

**CLO4 Alignment**: This notebook creates a practical security solution using data engineering and ML tools for a real-world scenario.


## 2) Environment Setup


In [ ]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

from joblib import dump

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

Path('models').mkdir(exist_ok=True)
Path('results').mkdir(exist_ok=True)


## 3) Dataset Loading & Initial Exploration


In [ ]:
DATA_PATH = Path('data/CIC_IDS2017.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(f'Loaded dataset: {DATA_PATH} with shape {df.shape}')
else:
    print('CIC_IDS2017.csv not found. Using synthetic fallback data for pipeline demonstration.')
    rng = np.random.default_rng(RANDOM_STATE)
    n = 4000
    labels = rng.choice(['BENIGN', 'DoS Hulk', 'FTP-Patator', 'SSH-Patator', 'Web Attack'], p=[0.7, 0.1, 0.06, 0.06, 0.08], size=n)
    df = pd.DataFrame({
        'Flow Duration': rng.normal(5e5, 2e5, n).clip(1),
        'Total Fwd Packets': rng.integers(1, 400, n),
        'Total Backward Packets': rng.integers(1, 400, n),
        'Total Length of Fwd Packets': rng.normal(8e4, 4e4, n).clip(0),
        'Flow Bytes/s': rng.normal(9e3, 5e3, n).clip(0),
        'Flow Packets/s': rng.normal(180, 80, n).clip(0),
        'protocol_type': rng.choice(['tcp', 'udp', 'icmp'], size=n, p=[0.7, 0.25, 0.05]),
        'service': rng.choice(['http', 'ssh', 'ftp', 'dns', 'other'], size=n),
        'flag': rng.choice(['SF', 'S0', 'REJ'], size=n),
        'Label': labels,
    })

print('Shape:', df.shape)
print('Columns (first 20):', list(df.columns[:20]))
print('Data types summary:')
print(df.dtypes.value_counts())
df.head()


## 4) Exploratory Data Analysis (EDA)


In [ ]:
if 'Label' not in df.columns:
    raise ValueError('Expected a target column named `Label` in CIC-IDS2017 data.')

label_counts = df['Label'].value_counts()
print('Attack/traffic distribution:')
print(label_counts)

plt.figure()
label_counts.head(15).plot(kind='bar')
plt.title('Top Traffic Labels by Count')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

binary_target = (df['Label'].astype(str).str.upper() != 'BENIGN').astype(int)
ratio = pd.Series(binary_target).value_counts().rename(index={0: 'Benign', 1: 'Malicious'})
print('Benign vs Malicious:')
print(ratio)

plt.figure()
ratio.plot(kind='pie', autopct='%1.1f%%')
plt.title('Benign vs Malicious Ratio')
plt.ylabel('')
plt.tight_layout()
plt.show()

missing = df.isna().sum()
print('Missing values (top 15):')
print(missing[missing > 0].sort_values(ascending=False).head(15) if (missing > 0).any() else 'No missing values detected.')

num_cols = df.select_dtypes(include=np.number).columns.tolist()
if num_cols:
    corr = df[num_cols[:25]].corr(numeric_only=True)
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, cmap='coolwarm', center=0)
    plt.title('Feature Correlation (subset)')
    plt.tight_layout()
    plt.show()


## 5) Data Preprocessing Pipeline


In [ ]:
df = df.drop_duplicates().copy()
df['target'] = (df['Label'].astype(str).str.upper() != 'BENIGN').astype(int)

X = df.drop(columns=['target'])
y = df['target']

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
if 'Label' in categorical_cols:
    categorical_cols.remove('Label')
if 'Label' in X.columns:
    X = X.drop(columns=['Label'])

numerical_cols = [c for c in X.columns if c not in categorical_cols]

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Class distribution (train):')
print(y_train.value_counts(normalize=True).rename({0: 'Benign', 1: 'Malicious'}))


## 6) Feature Engineering & Selection


In [ ]:
print(f'Numerical features: {len(numerical_cols)}')
print(f'Categorical features: {len(categorical_cols)}')
print('Example numerical features:', numerical_cols[:10])
print('Example categorical features:', categorical_cols[:10])


## 7) Model 1: Random Forest Classifier


In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', n_estimators=200, n_jobs=-1)),
])

rf_param_grid = {
    'model__max_depth': [None, 20],
    'model__min_samples_split': [2, 10],
}

rf_search = GridSearchCV(
    rf_pipeline,
    param_grid=rf_param_grid,
    scoring='recall',
    cv=3,
    n_jobs=-1,
    verbose=0,
)

start = time.time()
rf_search.fit(X_train, y_train)
rf_train_time = time.time() - start

rf_best = rf_search.best_estimator_
print('Best RF params:', rf_search.best_params_)
print(f'RF training time: {rf_train_time:.2f}s')


## 8) Model 2: Neural Network / Deep Learning


In [ ]:
nn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=50,
        random_state=RANDOM_STATE,
    )),
])

start = time.time()
nn_pipeline.fit(X_train, y_train)
nn_train_time = time.time() - start
print(f'Neural Network training time: {nn_train_time:.2f}s')


## 9) Model Evaluation - Random Forest


In [ ]:
def evaluate_binary_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = y_pred

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_score),
        'Specificity': specificity,
        'TP': tp,
        'TN': tn,
        'FP': fp,
        'FN': fn,
    }

    print(f"\n{name} Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious'], zero_division=0))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malicious'], yticklabels=['Benign', 'Malicious'])
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'results/confusion_matrix_{name.lower().replace(" ", "_")}.png', dpi=150)
    plt.show()

    return metrics, y_score

rf_metrics, rf_scores = evaluate_binary_model('Random Forest', rf_best, X_test, y_test)

rf_model = rf_best.named_steps['model']
rf_pre = rf_best.named_steps['preprocessor']
feature_names = rf_pre.get_feature_names_out()
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(20).sort_values().plot(kind='barh')
plt.title('Top 20 Random Forest Feature Importances')
plt.tight_layout()
plt.savefig('results/feature_importance_random_forest.png', dpi=150)
plt.show()


## 10) Model Evaluation - Neural Network


In [ ]:
nn_metrics, nn_scores = evaluate_binary_model('Neural Network', nn_pipeline, X_test, y_test)


## 11) Comparative Analysis


In [ ]:
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')),
])
baseline_pipeline.fit(X_train, y_train)
dt_metrics, dt_scores = evaluate_binary_model('Decision Tree', baseline_pipeline, X_test, y_test)

metrics_df = pd.DataFrame([rf_metrics, nn_metrics, dt_metrics])
metrics_df = metrics_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Specificity', 'FP', 'FN']]
metrics_df = metrics_df.sort_values(by='Recall', ascending=False)
metrics_df

metrics_df.to_csv('results/metrics_summary.csv', index=False)

plt.figure()
for model_name, scores in [('Random Forest', rf_scores), ('Neural Network', nn_scores), ('Decision Tree', dt_scores)]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc_val = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.6)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.tight_layout()
plt.savefig('results/roc_curves.png', dpi=150)
plt.show()

if 'Label' in df.columns:
    X_all = df.drop(columns=['target'])
    y_all = df['target']
    labels_all = df['Label'].astype(str)

    attack_df = pd.DataFrame({'Label': labels_all, 'ActualMalicious': y_all})
    preds_all = rf_best.predict(X_all.drop(columns=['Label']) if 'Label' in X_all.columns else X_all)
    attack_df['PredMalicious_RF'] = preds_all

    per_attack = (
        attack_df[attack_df['Label'].str.upper() != 'BENIGN']
        .groupby('Label')
        .apply(lambda g: pd.Series({'DetectionRate_RF': (g['PredMalicious_RF'] == 1).mean(), 'Count': len(g)}))
        .sort_values('DetectionRate_RF', ascending=False)
    )
    per_attack
    per_attack.to_csv('results/attack_type_detection_random_forest.csv')


## 12) Security Analysis & Interpretation

- **False Negatives (FN)** are critical in intrusion detection because missed malicious traffic can permit compromise.
- **False Positives (FP)** increase alert fatigue and SOC workload.
- Deployment should tune decision thresholds to maximize recall for high-risk attack classes while controlling operational noise.
- Feature importance supports mapping model behavior to likely indicators of compromise.


## 13) Model Serialization & Artifacts


In [ ]:
dump(rf_best, 'models/random_forest_model.joblib')
dump(nn_pipeline, 'models/neural_network_model.joblib')
dump(baseline_pipeline, 'models/decision_tree_model.joblib')

print('Saved model artifacts to models/')
print('Saved evaluation artifacts to results/')


## 14) Conclusion & Future Enhancements

This proof-of-concept demonstrates behavior-based IDS capabilities for SecureNet Corp.

### Future enhancements
- Full-scale CIC-IDS2017 multi-file ingestion and attack-family tuning
- More advanced deep learning/ensemble approaches
- Streaming inference integration with SIEM/SOAR
- Drift monitoring and scheduled retraining
